# ENGR 1451/2451 — Weekly Assignment 7
**10 points · 4 graded parts**  
**Due:** See Canvas.

Practice confounding, standard error, and confidence intervals. Show your Python code and keep written answers brief.


## What to hand in

Work with the **course tutor** in the class project. Use it as your AI help; it keeps the record, so no separate log is needed. Your code and written answers must still be your own.

At the end of each working session, type **"make my record"** in the chat. Copy its block into **Session records** below, placing each new record after the previous one.

To continue later, start a new chat in the same project and state the assignment and where you stopped. Session records are not graded for correctness or how much help you needed.


## Setup

All data are embedded; no additional files are needed. Keep full precision in calculations and round only displayed results.


In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

pd.set_option("display.precision", 4)


## LE7-1. Group composition and confounding (4 pts)

These synthetic inspection records compare two manufacturing methods. Methods were **not randomly assigned** to parts.

Calculate pass rates for each method within each part type and overall. For overall rates, use `groupby()` and sum the counts before dividing; do not average percentages.

Which method has the higher overall rate? Which has the higher rate within each part type? In two sentences, explain the reversal using group composition and why these observations do not establish a causal effect of method.

**Answer:**
Method B has the overall higher pass rate. Method A has the higher pass rate within the simple and complex. Method B has more simple than complex which makes the data appear this way. They method weren't randomly distributed so tis is the confounding factor, therefore the observations don't establish any conclusions

In [2]:
inspection = pd.DataFrame({
    "method": ["A", "A", "B", "B"],
    "part_type": ["Simple", "Complex", "Simple", "Complex"],
    "inspected": [20, 80, 80, 20],
    "passed": [18, 48, 64, 10],
})

# Calculate and display pass rates within each part type.
# Group by method, sum inspected/passed counts, and calculate overall rates.
inspection["pass_rate"] = inspection["passed"]/inspection["inspected"]
display(
    inspection[["method", "part_type", "inspected", "passed", "pass_rate"]]
)
overall = inspection.groupby("method")[["inspected", "passed"]].sum()
overall["pass_rate"] = overall["passed"] / overall["inspected"]

display(overall)

,method,part_type,inspected,passed,pass_rate
0,A,Simple,20,18,0.9
1,A,Complex,80,48,0.6
2,B,Simple,80,64,0.8
3,B,Complex,20,10,0.5


,inspected,passed,pass_rate
method,,,
A,100,66,0.66
B,100,74,0.74


## LE7-2. Estimate a population mean (4 pts)

The synthetic tensile strengths below are in **MPa**. Assume an independent random sample from a stable, approximately normal production process. Its population mean and SD are unknown.

### A. Estimate the SD and SE (2 pts)

Display $n$, $\bar x$, sample SD $s$ using `std(ddof=1)`, and estimated SE using `stats.sem(..., ddof=1)`.

In one sentence, distinguish what $s$ and $s/\sqrt{n}$ measure.

**Answer:**
The sample sd shows the spread of individual observations. S/sqrt(n) shows the standard error which is variability of the sample mean. increasing n reduces error but does not effect the sample SD

In [3]:
strength = np.array([
    48.2, 51.4, 49.7, 52.1, 46.8, 50.5, 53.0, 47.6,
    49.1, 51.8, 48.9, 50.7, 47.9, 52.5, 49.8, 50.2,
])

# Calculate and display n, sample_mean, sample_sd, and estimated_se.
n = len(strength)
sample_mean = strength.mean()
sample_sd = strength.std(ddof=1)
estimated_se = stats.sem(strength, ddof=1)

print("n =", n)
print("sample mean =", sample_mean)
print("sample sd =", sample_sd)
print("estimated se =", estimated_se)

n = 16
sample mean = 50.0125
sample sd = 1.8518009252256753
estimated se = 0.4629502313064188


### B. Construct and interpret a confidence interval (2 pts)

Because $\sigma$ is unknown, use a **Student's $t$ interval**:

$$\bar x \pm t^*\frac{s}{\sqrt{n}},\qquad df=n-1.$$

Calculate `t_star = stats.t.ppf(0.975, df=n-1)` for 95% confidence. Display $df$, $t^*$, and both interval endpoints in MPa.

Interpret the interval for the **population mean**. Does it describe where 95% of individual strengths fall? Would a 99% interval using the same sample be wider or narrower? No second calculation is required.

**Answer:**
There is a 95% CI that the population mean strength is between 49-51. It measures rhe pop. mean not the range of observations that fall within it. A 99% interval would be wider because it requires a different value.

In [5]:
# Calculate the t critical value and margin of error.
# Display df, t_star, and the lower/upper confidence limits in MPa.
df = n -1
t_star = stats.t.ppf(0.975, df=df)
marginE = t_star*estimated_se

lower = sample_mean - marginE
upper = sample_mean + marginE


print("df =", df)
print("t* =", t_star)
print("Lower limit =", lower, "MPa")
print("Upper limit =", upper, "MPa")

df = 15
t* = 2.131449545559323
Lower limit = 49.025744939865355 MPa
Upper limit = 50.99925506013465 MPa


## LE7-3. Estimate a population percentage (2 pts)

*Adapted from Chapter 21, Review Exercise 2 (printed p. 391; PDF p. 17).*

In the chapter's historical survey, a simple random sample of **500 households** from a town of **25,000 households** included **239 with internet access**.

Use Python to calculate the estimated proportion and its estimated SE:

$$\hat p=\frac{x}{n},\qquad
\widehat{\operatorname{SE}}(\hat p)=\sqrt{\frac{\hat p(1-\hat p)}{n}}.$$

For this large-sample proportion, use the normal approximation with `z_star = stats.norm.ppf(0.975)`:

$$\hat p\pm z^*\widehat{\operatorname{SE}}(\hat p).$$

Ignore the finite-population correction; the sample includes only 2% of the town's households.

Display the estimated **percentage**, its SE in **percentage points**, and the approximate **95% confidence interval as percentages**. Interpret the interval for all households in the town in one sentence.

**Answer:**
We are 95% confident that somewhere between 43.4% and 52.2% of homes have internet

In [6]:
total_households = 25_000
sampled_households = 500
with_internet = 239

# Calculate p_hat and its estimated SE using np.sqrt().
# Find z_star and calculate the lower and upper confidence limits.
# Display the estimate, SE, and interval in percentage units.

p_hat = with_internet / sampled_households

se = np.sqrt(
    p_hat * (1 - p_hat) / sampled_households
)

z_star = stats.norm.ppf(0.975)

lower = p_hat - z_star * se
upper = p_hat + z_star * se

print("estimated percent ", p_hat * 100, )
print("se =", se * 100, "percent points")
print("95% CI =", lower * 100, "% to", upper * 100, )

estimated percent  47.8
se = 2.23390241505756 percent points
95% CI = 43.42163172151013 % to 52.17836827848986


## Session records

Paste the block from **"make my record"** after each working session. Keep all session records in order.


<!-- Paste your session record blocks below, one after another. -->

*(session records go here)*
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA7",
  "export_n": 1,
  "session_date": "2026-10-20",
  "messages": [
    {"i": 1, "who": "student", "text": "i am ready to start"},
    {"i": 2, "who": "tutor", "served": ["how_to_use", "a7_preamble", "a7_setup", "a7_le7_1"]},
    {"i": 3, "who": "student", "text": "how should I start"},
    {"i": 4, "who": "tutor", "text": "A good place to start is the first display LE7-1 asks for: a pass rate for each of the four rows of `inspection`. Each row already holds one method within one part type, so each rate is a single division, and you can do all four rows at once.\n\nTwo quick pieces of syntax that may help:\n\n- Dividing one column by another works row by row, and you can store the result as a new column. The generic shape is `df[\"rate\"] = df[\"num\"] / df[\"den\"]`.\n- Use `/`, not `//`. Floor division would turn every rate below 1 into 0.\n\nSo the first question for you: which of the two count columns is the numerator and which is the denominator for a pass rate?\n\nOnce you have the four rates displaying, the second move is the overall rate for each method. That one uses `groupby()` and summed counts, and I am glad to explain how it works whenever you want, including why averaging the within-type percentages would give a different answer.\n\nTry the within-type rates and post your cell here. You have a clear plan already, since the notebook's comments spell out the steps, so you are in good shape to begin."},
    {"i": 5, "who": "student", "text": "okay so use the insepection  pass rate and set it equal to the passed/inpecred and then use the groupby command to find the overall"},
    {"i": 6, "who": "tutor", "text": "[trimmed] Confirmed the plan for the pass-rate column; explained how `groupby()` splits rows by a key and `.sum()` collapses each group, with the generic shape `totals = df.groupby(\"key\")[[\"den\", \"num\"]].sum()` and `totals[\"num\"] / totals[\"den\"]`; warned about double brackets and about summing a rate column; asked which column to group by."},
    {"i": 7, "who": "student", "text": "is df just a placeholder"},
    {"i": 8, "who": "tutor", "text": "[trimmed] Yes, `df` is a placeholder for a DataFrame; in the notebook it is `inspection`. Same for \"key\", \"num\", \"den\", \"rate\". Gave a toy `clinic` example of a row-by-row rate column."},
    {"i": 9, "who": "student", "text": "[image: LE7-1 code cell with pass_rate column, display of the four rows, overall = inspection.groupby(\"method\")[[\"inspected\", \"passed\"]].sum(), overall[\"pass_rate\"] = overall[\"passed\"] / overall[\"inspected\"], display(overall); output shows rates 0.9, 0.6, 0.8, 0.5 and overall 0.66 and 0.74] how dooes this look", "code": true},
    {"i": 10, "who": "tutor", "text": "[trimmed] Confirmed the code meets the LE7-1 requirements (within-type rates, grouped summed counts divided after, both displayed). Listed what the Answer field must cover: the comparisons, the group-composition explanation, and why the data are not causal."},
    {"i": 11, "who": "student", "text": "I am going to do all written parts at the end"},
    {"i": 12, "who": "tutor", "text": "That works. Your LE7-1 code is done, and you can leave the **Answer:** fields until the end. One practical note: if you restart the kernel before you get back to them, run the import cell again first so `np`, `pd`, and `stats` exist.\n\nDo you want to move on to LE7-2, or stop here? If you want LE7-2, say the word and I will post it."},
    {"i": 13, "who": "student", "text": "ready to move on"},
    {"i": 14, "who": "tutor", "served": ["a7_le7_2"]},
    {"i": 15, "who": "student", "text": "okay so just find the mean std and standard erroe using the stats function"},
    {"i": 16, "who": "tutor", "text": "[trimmed] Clarified that mean and SD are array methods (`values.mean()`, `values.std(ddof=1)`) and the SE comes from `stats.sem(values, ddof=1)`; explained what `ddof` changes and that NumPy's default is 0; recommended `len()` for `n` and using the names from the cell's comment."},
    {"i": 17, "who": "student", "text": "do i  need to find the t star value"},
    {"i": 18, "who": "tutor", "text": "[trimmed] Not in part A; t* belongs to part B, where the prompt gives the line. Part B reads `n` from part A."},
    {"i": 19, "who": "student", "text": "[image: LE7-2A cell with n = len(strength), sample_mean = strength.mean(), sample_sd = strength.std(ddof=1), estimated_se = stats.sem(strength, ddof=1), four labeled prints; output shown] this is what i did", "code": true},
    {"i": 20, "who": "tutor", "text": "[trimmed] Confirmed part A code meets the prompt (calculated n, ddof=1, stats.sem, no early rounding, all four displayed). Suggested an optional check that s/√n matches the SE and listed what the one-sentence Answer should address."},
    {"i": 21, "who": "student", "text": "so do I need to change any code"},
    {"i": 22, "who": "tutor", "text": "No, your part A code does not need any changes. Part B reads `n`, `sample_mean`, `sample_sd`, and `estimated_se` from this cell, so it must be run first, and the one-sentence Answer is still open."},
    {"i": 23, "who": "student", "text": "yes ready to move on"},
    {"i": 24, "who": "tutor", "text": "[trimmed] Restated part B and outlined its four code pieces (df, t_star, margin, endpoints); explained the margin and endpoints shapes and why 0.975 is used for a 95% interval."},
    {"i": 25, "who": "student", "text": "How should I start this part"},
    {"i": 26, "who": "tutor", "text": "[trimmed] Gave a five-step order: df, t_star line as printed, margin from the SE, endpoints, display with MPa units; asked the student to try the first two and post."},
    {"i": 27, "who": "student", "text": "okay set df as n-1 and then use the stats.t"},
    {"i": 28, "who": "tutor", "text": "[trimmed] Confirmed the start; explained `stats.t`, `.ppf`, and `df=`, and warned that the first argument is a probability (0.975, not 97.5 or 0.95)."},
    {"i": 29, "who": "student", "text": "and from that info I can find the margin error and upper/lower bounds"},
    {"i": 30, "who": "tutor", "text": "[trimmed] Confirmed: margin = t_star times estimated_se; lower and upper from sample_mean; keep full precision and round only in the display; show df, t_star, and the limits with MPa."},
    {"i": 31, "who": "student", "text": "[image: LE7-2B cell with df = n - 1, t_star = stats.t.ppf(0.975, df=df), marginE = t_star*estimated_se, lower and upper from sample_mean, four prints with MPa; output shown] how does this look", "code": true},
    {"i": 32, "who": "tutor", "text": "[trimmed] Confirmed part B code meets the prompt and needs no changes. Listed what the three written answers must address: interpretation for the population mean, whether it describes individual strengths, and 99% wider or narrower."},
    {"i": 33, "who": "student", "text": "ready for the next"},
    {"i": 34, "who": "tutor", "served": ["a7_le7_3"]},
    {"i": 35, "who": "student", "text": "do i just need to find the standard error upper/lower and

## Before you submit

1. **Restart Kernel and Run All Cells**; confirm there are no errors.
2. Check that calculations, tables, and written answers are visible.
3. Include your session record block(s).
4. Save and submit the completed `.ipynb` file in Canvas.
